# Launching ND AI Lab on the pollen project

The same thing `launch_nd_ai_lab.py` does, from a notebook, pointed at
`data/pollen_count`.

The project is used **in place**. It already holds labels, predictions and
models, and anything you annotate here is written back into it.

## Qt event loop

napari needs a Qt event loop. In a notebook that comes from `%gui qt`,
not from `napari.run()` -- calling `run()` here would block the kernel.

In [1]:
%gui qt

## Find the project

Works whether the kernel starts in `notebooks/` or at the repo root.

In [2]:
from pathlib import Path

for candidate in (Path.cwd() / "data" / "pollen_count",
                  Path.cwd() / "notebooks" / "data" / "pollen_count"):
    if candidate.is_dir():
        project = candidate
        break
else:
    raise FileNotFoundError("pollen_count not found from %s" % Path.cwd())

print(project)
print(len(sorted(project.glob("*.png"))), "images")

C:\Users\bnort\work\ImageJ2022\tnia\i2k-2026\notebooks\data\pollen_count
10 images


## Launch

`register_all=True` registers every segmenter and augmenter, so the
dropdowns are populated. Pass `False` to register only what you import
yourself.

In [3]:
import napari
from napari_ai_lab.apps.nd_ai_lab_launcher import launch_nd_ai_lab

viewer = napari.Viewer()

ai_lab, sequence_viewer, model = launch_nd_ai_lab(
    viewer,
    project,
    viewer_type="sequence",
    axes_to_collapse="C",
    axis_types="NYXC",
    register_all=True,
)

print("ND AI Lab launched on", project.name)

INFO:OpenGL.acceleratesupport:No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


Registered global segmenter: StarDist2D (scikit-ops)
Registered global segmenter: Cellpose3 (scikit-ops)
Registered global segmenter: Cellpose4 (scikit-ops)
Registered global segmenter: CellCastStardistSegmenter
Registered global segmenter: ThresholdSegmenter
Registered global segmenter: MicroSamSegmenter
Registered global segmenter: MonaiUNetSegmenter
Registered global segmenter: MonaiUNetSegmenter3D
Registered global segmenter: MicrosamYoloSegmenter
Registered global segmenter: SkImageWatershedSegmenter
Registered interactive segmenter: Otsu2D
Registered interactive segmenter: Otsu3D
Registered interactive segmenter: SAM3D
Registered interactive segmenter: SAMSphere3D
Registered interactive segmenter: RegionGrow3D
Registered interactive segmenter: FeatureRegionGrow3D
Registered interactive segmenter: AnisotropicSphereFit3D
Registered interactive segmenter: HoughSphereFit3D
Registered augmenter: SimpleAugmenter
Registered augmenter: AlbumentationsAugmenter
Connected to viewer close ev

## Why those arguments

| argument | why |
|---|---|
| `viewer_type="sequence"` | the ten images have different shapes, so they cannot be one stacked array |
| `axes_to_collapse="C"` | colour is not an axis to annotate along; labels are 2D per image |
| `axis_types="NYXC"` | N images, each Y by X with a colour axis |

A project of equal-sized images could use `viewer_type="stacked"` instead.

## Using it

The **Sequence Viewer** at the bottom moves between the ten images. The
**AI Lab** dock on the right has Label, Augment and Segment.

Interactive segmentation lives on the Label tab. SAM3D there needs
micro_sam, which the pixi environment has and the pip fallback does not.